# End-to-end example: Drell-Yan at the LHC

Take **pp → μ⁺μ⁻** from diagram → tree |M̄|² → partonic σ̂ → hadronic σ(pp) →
NLO K-factor → PDF band → fiducial σ → dσ/dM_ll, plus the related decay Z→μ⁺μ⁻.

**Install** (see [INSTALLATION.md](../INSTALLATION.md)):

```bash
pip install feynman-engine
feynman setup --profile lhc-analyst   # builds QGRAF, FORM, LoopTools, LHAPDF + textbook + all-lhc-nlo OL packs
feynman doctor                        # verify
```

If you just want this notebook to run, `feynman setup --profile student`
suffices (only the textbook pack needed for DY).

## 1. Born diagram

In [ ]:
import warnings, os
# Mute OpenLoops/LHAPDF stderr noise that isn't actionable here.
warnings.filterwarnings("ignore")
os.environ.setdefault("OPENLOOPS_VERBOSE", "0")

from feynman_engine.engine import FeynmanEngine
engine = FeynmanEngine()
result = engine.generate("e+ e- -> mu+ mu-", theory="QED", loops=0, output_format="svg")
print(f"{result.summary['total_diagrams']} diagram(s); first topology: {result.diagrams[0].topology}")

## 2. Spin-averaged |M̄|² (γ + Z, partonic)

In [ ]:
from feynman_engine.physics.amplitude import get_curated_amplitude
entry = get_curated_amplitude("e+ e- -> mu+ mu-", "EW")
print(entry.description)
print("\n|M̄|² =", str(entry.msq)[:240], "...")

## 3. Partonic σ̂(e⁺e⁻ → μ⁺μ⁻) on the Z peak

In [ ]:
from feynman_engine.amplitudes.cross_section import total_cross_section
r = total_cross_section("e+ e- -> mu+ mu-", "EW", sqrt_s=91.2)
print(f"σ(e⁺e⁻ → μ⁺μ⁻) at √s = 91.2 GeV (Z peak): {r['sigma_pb']:.2f} pb")
# Off-peak for comparison
r2 = total_cross_section("e+ e- -> mu+ mu-", "EW", sqrt_s=200.0)
print(f"σ(e⁺e⁻ → μ⁺μ⁻) at √s = 200 GeV          : {r2['sigma_pb']:.3f} pb")

## 4. Hadronic σ(pp → μ⁺μ⁻) at 13 TeV (NNPDF4.0 LO bundled)

In [ ]:
from feynman_engine.amplitudes.hadronic import hadronic_cross_section
r = hadronic_cross_section(
    "p p -> mu+ mu-", sqrt_s=13000.0, theory="EW", order="LO",
    m_ll_min=66.0, m_ll_max=116.0,
)
print(f"σ(DY, M_ll∈[66,116]) = {r['sigma_pb']:.0f} pb   (CMS ≈ 1900 pb at NLO)")
print(f"PDF: {r['pdf']}   μ_F = {r['mu_f_gev']:.1f} GeV")

## 5. NLO K-factor (tabulated K × LO PDF)

In [ ]:
r_nlo = hadronic_cross_section(
    "p p -> mu+ mu-", sqrt_s=13000.0, theory="EW", order="NLO",
    m_ll_min=66.0, m_ll_max=116.0,
)
print(f"σ_NLO = {r_nlo['sigma_pb']:.0f} pb   K-factor = {r_nlo['k_factor']:.2f}")

## 6. PDF uncertainty band (5 NNPDF replicas)

In [ ]:
from feynman_engine.amplitudes.hadronic import hadronic_cross_section_pdf_uncertainty
r = hadronic_cross_section_pdf_uncertainty(
    "p p -> mu+ mu-", sqrt_s=13000.0, theory="EW", order="LO",
    m_ll_min=66.0, m_ll_max=116.0, n_members=5, member_stride=20,
)
sig, up, dn, rel = r['sigma_pb'], r['sigma_pb_pdf_up'], r['sigma_pb_pdf_down'], r['sigma_pb_pdf_relative']
print(f"σ = {sig:.0f}  +{up - sig:.0f} / -{sig - dn:.0f} pb   (±{rel*100:.1f}%)")

## 7. CMS-style fiducial cuts (pT_l > 25, |η_l| < 2.4)

In [ ]:
r_fid = hadronic_cross_section(
    "p p -> mu+ mu-", sqrt_s=13000.0, theory="EW", order="LO",
    m_ll_min=66.0, m_ll_max=116.0, pT_l_min=25.0, eta_l_max=2.4,
)
print(f"σ fiducial = {r_fid['sigma_pb']:.0f} pb   acceptance = {r_fid['sigma_pb']/r['sigma_pb']*100:.0f}%")

## 8. dσ/dM_ll across the Z peak

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from feynman_engine.amplitudes.differential import hadronic_differential_distribution
edges = np.linspace(60, 120, 13)
h = hadronic_differential_distribution(
    "p p -> mu+ mu-", sqrt_s=13000.0, observable="M_ll",
    bin_edges=edges.tolist(), theory="EW", n_events=20_000,
)
plt.figure(figsize=(7,4))
plt.bar(h['bin_centers'], h['dsigma_dX_pb'], width=np.diff(edges)*0.9, color='steelblue', edgecolor='k')
plt.axvline(91.19, color='r', ls='--', label='M_Z')
plt.xlabel(r"$M_{\ell\ell}$ [GeV]"); plt.ylabel(r"$d\sigma/dM$ [pb/GeV]")
plt.title(f"pp → μ⁺μ⁻ at 13 TeV  (σ_total = {h['sigma_total_pb']:.0f} pb)")
plt.legend(); plt.tight_layout(); plt.show()

## 9. Same engine for Z → μ⁺μ⁻ decay width

In [ ]:
from fastapi.testclient import TestClient
from feynman_engine.api.app import app
client = TestClient(app)
w = client.get("/api/amplitude/decay-width",
               params={"process": "Z -> mu+ mu-", "theory": "EW"}).json()
print(f"Γ(Z → μ⁺μ⁻) = {w['width_mev']:.2f} MeV   (PDG: {w.get('pdg_width_mev','?')} MeV)")

## What's covered

Tree diagrams • partonic |M̄|² • partonic σ̂ • PDF convolution • NLO
K-factor • PDF uncertainty • fiducial cuts • differential distributions •
decay widths.  The same surface works for any of the ~200 curated processes
listed at `/api/amplitude/processes`.